<a href="https://colab.research.google.com/github/mprasads5/NASSCOM_VEDIC_Schedule1/blob/main/U8_%E2%80%94_Data_Cleaning_%26_Preprocessing_(Part_2)_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Core imports for the whole lab
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)

Setup complete. pandas 2.2.2


In [2]:
# -----------------------------------------------------------
# A SMALL, ALREADY-CLEAN DATASET (Part 1 did the cleaning)
# -----------------------------------------------------------
# Mixed columns: an unordered category (city), an ordered category
# (size), two numeric features on very different scales, and a target.
df = pd.DataFrame({
    'city':   ['pune', 'delhi', 'mumbai', 'pune', 'delhi', 'mumbai', 'pune', 'delhi'],
    'size':   ['small', 'large', 'medium', 'medium', 'small', 'large', 'large', 'small'],
    'age':    [25, 41, 33, 29, 52, 38, 46, 22],
    'income': [38000, 92000, 55000, 47000, 120000, 76000, 88000, 41000],
    'bought': [0, 1, 0, 0, 1, 1, 1, 0],   # target
})
df

,city,size,age,income,bought
0,pune,small,25,38000,0
1,delhi,large,41,92000,1
2,mumbai,medium,33,55000,0
3,pune,medium,29,47000,0
4,delhi,small,52,120000,1
5,mumbai,large,38,76000,1
6,pune,large,46,88000,1
7,delhi,small,22,41000,0


In [3]:
# -----------------------------------------------------------
# 🔹 1A. ONE-HOT ENCODING (for UNORDERED categories)
# -----------------------------------------------------------

# 'city' has no natural order -> one 0/1 column per category
city_ohe = pd.get_dummies(df['city'], prefix='city').astype(int)
city_ohe

,city_delhi,city_mumbai,city_pune
0,0,0,1
1,1,0,0
2,0,1,0
3,0,0,1
4,1,0,0
5,0,1,0
6,0,0,1
7,1,0,0


In [4]:
# -----------------------------------------------------------
# 🔹 1B. LABEL / ORDINAL ENCODING (for ORDERED categories)
# -----------------------------------------------------------

# 'size' has a real order: small < medium < large -> map to 0,1,2
size_order = {'small': 0, 'medium': 1, 'large': 2}
df['size_code'] = df['size'].map(size_order)
df[['size', 'size_code']]

,size,size_code
0,small,0
1,large,2
2,medium,1
3,medium,1
4,small,0
5,large,2
6,large,2
7,small,0


In [5]:
# 1. one-hot encode just the 'city' column of the whole df
city_ohe = pd.get_dummies(df['city'], prefix='city').astype(int)
print(city_ohe.head())

# 2. ordinal-encode 'size' (map with size_order)
df['size_code'] = df['size'].map(size_order)
print(df[['size', 'size_code']].head())

# 3. Why one-hot is wrong for size / ordinal is wrong for city:
# One-hot is wrong for size because size has a natural order
# (small < medium < large) that would be lost.
# Ordinal encoding is wrong for city because cities have no
# natural order; assigning numbers like Pune=0, Delhi=1,
# Mumbai=2 would incorrectly imply a ranking or distance.

   city_delhi  city_mumbai  city_pune
0           0            0          1
1           1            0          0
2           0            1          0
3           0            0          1
4           1            0          0
     size  size_code
0   small          0
1   large          2
2  medium          1
3  medium          1
4   small          0


In [6]:
# -----------------------------------------------------------
# 🔹 2A. THE PROBLEM — FEATURES ON DIFFERENT SCALES
# -----------------------------------------------------------

# income (tens of thousands) dwarfs age (tens). A distance-based
# model would treat income as nearly all that matters.
print(df[['age', 'income']].describe().loc[['min', 'max', 'mean']])

        age    income
min   22.00   38000.0
max   52.00  120000.0
mean  35.75   69625.0


In [7]:
# -----------------------------------------------------------
# 🔹 2B. STANDARDISATION (Z-score)  vs  NORMALISATION (Min-Max)
# -----------------------------------------------------------
from sklearn.preprocessing import StandardScaler, MinMaxScaler

num = df[['age', 'income']]

z = StandardScaler().fit_transform(num)        # mean 0, std 1
m = MinMaxScaler().fit_transform(num)          # range [0, 1]

print('Standardised (mean~0, std~1):')
print(pd.DataFrame(z, columns=['age', 'income']).round(2).head(3))
print('\nMin-Max (range 0..1):')
print(pd.DataFrame(m, columns=['age', 'income']).round(2).head(3))

Standardised (mean~0, std~1):
    age  income
0 -1.10   -1.16
1  0.54    0.82
2 -0.28   -0.54

Min-Max (range 0..1):
    age  income
0  0.10    0.00
1  0.63    0.66
2  0.37    0.21


In [10]:
income = df[['income']]   # 2D shape for sklearn

# 1. standardise -> check mean ~0, std ~1
z = StandardScaler().fit_transform(income)
print('Standardised (mean~0, std~1):')
print(pd.DataFrame(z, columns=['income']).round(2).head(3))

# 2. min-max scale -> check min 0, max 1
m = MinMaxScaler().fit_transform(income)          # range [0, 1]
print('\nMin-Max (range 0..1):')
print(pd.DataFrame(m, columns=['income']).round(2).head(3))

# 3. Needs scaling: ...   Doesn't need it: ...   (comment)
#  Needs scaling: income, salary, age, spend, marks, height, weight
# Doesn't need it: city, gender, department, country, category (categorical features)

Standardised (mean~0, std~1):
   income
0   -1.16
1    0.82
2   -0.54

Min-Max (range 0..1):
   income
0    0.00
1    0.66
2    0.21


In [11]:
# -----------------------------------------------------------
# 🔹 3A. SPLIT FIRST, THEN FIT ON TRAIN ONLY
# -----------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[['age', 'income']]
y = df['bought']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # FIT + transform on train
X_test_s  = scaler.transform(X_test)        # only TRANSFORM on test
print('train rows:', X_train.shape[0], '| test rows:', X_test.shape[0])
print('scaler learned mean from TRAIN only:', scaler.mean_.round(1))

train rows: 6 | test rows: 2
scaler learned mean from TRAIN only: [3.45e+01 6.90e+04]


In [14]:
# 1a. WRONG: fit on ALL data, then split (leakage!)
wrong_mean = StandardScaler().fit(X).mean_

# 1b. RIGHT: fit on TRAIN only
right_mean = StandardScaler().fit(X_train).mean_

print('fit-on-all mean :', wrong_mean.round(1))
print('fit-on-train mean:', right_mean.round(1))

# 2. Why fitting on all data is a problem: ...   (write your answer)
# Fitting on all data causes data leakage.
# The scaler learns information from the test set before model evaluation.
# As a result, the model indirectly "sees" the test data during training.
# This can produce overly optimistic accuracy and unreliable performance estimates.
# Therefore, preprocessing steps should always be fitted only on the training data.

fit-on-all mean : [3.5800e+01 6.9625e+04]
fit-on-train mean: [3.45e+01 6.90e+04]


In [15]:
# -----------------------------------------------------------
# 🔹 4A. COMBINE & BIN EXISTING COLUMNS
# -----------------------------------------------------------

fe = df.copy()
# combine: income per year of age (a crude 'earning rate')
fe['income_per_age'] = (fe['income'] / fe['age']).round(0)
# bin: turn continuous age into life-stage buckets
fe['age_group'] = pd.cut(fe['age'], bins=[0, 30, 45, 100],
                         labels=['young', 'mid', 'senior'])
fe[['age', 'income', 'income_per_age', 'age_group']]

,age,income,income_per_age,age_group
0,25,38000,1520.0,young
1,41,92000,2244.0,mid
2,33,55000,1667.0,mid
3,29,47000,1621.0,young
4,52,120000,2308.0,senior
5,38,76000,2000.0,mid
6,46,88000,1913.0,senior
7,22,41000,1864.0,young


In [16]:
# -----------------------------------------------------------
# 🔹 4B. EXTRACT FROM A DATE
# -----------------------------------------------------------

dates = pd.to_datetime(['2024-01-06', '2024-03-15', '2024-07-21', '2024-12-25'])
d = pd.DataFrame({'date': dates})
d['day_of_week'] = d['date'].dt.day_name()   # Monday, Tuesday, ...
d['month']       = d['date'].dt.month
d['is_weekend']  = d['date'].dt.dayofweek >= 5
d

,date,day_of_week,month,is_weekend
0,2024-01-06,Saturday,1,True
1,2024-03-15,Friday,3,False
2,2024-07-21,Sunday,7,True
3,2024-12-25,Wednesday,12,False


In [17]:
ex = df.copy()

# 1. high_earner flag (income > median)
median_income = ex['income'].median()
ex['high_earner'] = ex['income'] > median_income

# 2. bin income into 3 buckets with pd.cut
ex['income_group'] = pd.cut(
    ex['income'],
    bins=3,
    labels=['Low', 'Medium', 'High']
)

# 3. show income + the new columns
print(ex[['income', 'high_earner', 'income_group']])

   income  high_earner income_group
0   38000        False          Low
1   92000         True       Medium
2   55000        False          Low
3   47000        False          Low
4  120000         True         High
5   76000         True       Medium
6   88000         True       Medium
7   41000        False          Low


In [18]:
# -----------------------------------------------------------
# 🔹 5A. ONE LEAK-FREE PIPELINE: PREPROCESS + MODEL
# -----------------------------------------------------------
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

num_cols = ['age', 'income']
cat_cols = ['city', 'size']

# scale the numbers, one-hot the categories — all in one object
pre = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])
pipe = Pipeline([('prep', pre), ('model', LogisticRegression(max_iter=1000))])
print(pipe)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'income']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['city', 'size'])])),
                ('model', LogisticRegression(max_iter=1000))])
